In [21]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [22]:
import csv
import json
import os
import sys
import time
from datetime import date, timedelta
from loguru import logger
from dotenv import load_dotenv
from google.cloud import storage
from utils.api_utils import WeatherAPI
from config import GCS_BUCKET
import sys, os
from pyspark.sql.functions import col, from_json, to_timestamp,lit,concat,now,date_format,current_timestamp,date_trunc,from_unixtime,round,avg,sum  
load_dotenv()



True

In [23]:
from spark_session import get_spark
spark = get_spark("Silver Processing")

In [24]:
weather_bronze_path = f"gs://{GCS_BUCKET}/bronze/weather/2026-01-05/"
traffic_bronze_path = f"gs://{GCS_BUCKET}/bronze/traffic/2026-05-04/"

In [25]:
weather_df = spark.read.format("delta").load(weather_bronze_path)


In [26]:
weather_df.show()

+----------------+--------+--------------------+-----------+-------------+---------+-------------+
|        datetime| road_id|           road_name|temperature|      weather|windspeed|precipitation|
+----------------+--------+--------------------+-----------+-------------+---------+-------------+
|2026-01-05T00:00|28207692|Đường Hoàng Quốc ...|       17.0|     Overcast|      7.2|          0.0|
|2026-01-05T01:00|28207692|Đường Hoàng Quốc ...|       17.0|     Overcast|      7.6|          0.0|
|2026-01-05T02:00|28207692|Đường Hoàng Quốc ...|       17.1|Light drizzle|      8.1|          0.2|
|2026-01-05T03:00|28207692|Đường Hoàng Quốc ...|       17.0|Light drizzle|      7.6|          0.1|
|2026-01-05T04:00|28207692|Đường Hoàng Quốc ...|       16.8|Light drizzle|      8.5|          0.2|
|2026-01-05T05:00|28207692|Đường Hoàng Quốc ...|       16.7|Light drizzle|     10.9|          0.2|
|2026-01-05T06:00|28207692|Đường Hoàng Quốc ...|       16.3|Light drizzle|     15.0|          0.3|
|2026-01-0

In [27]:
weather_df.groupBy("road_name").count().show()

+--------------------+-----+
|           road_name|count|
+--------------------+-----+
|Ngõ 107 Trần Quốc...|   24|
|Ngõ 61 Phố Phạm T...|   24|
|Ngõ 238 Hoàng Quố...|   24|
|Ngõ 1 Phố Dịch Vọ...|   24|
|Ngách 41/6 Trần Q...|   24|
|Ngõ 68 Đường Cầu ...|   24|
|Ngõ 20 Phố Trần Q...|   24|
|Ngõ 25 Phố Hoàng ...|   24|
|Ngõ 387 - Hoàng Q...|   24|
|Ngõ 2 Phố Trần Qu...|   24|
|Ngõ 49 Phố Trần Đ...|   24|
|Ngõ 130 Đường Xuâ...|   24|
|Đường Nguyễn Phon...|   24|
|Đường Hoàng Quốc ...|   24|
|Ngõ 12 Phố Trần Q...|   24|
|Ngõ 30 Phố Trần Q...|   24|
|Đường Nguyễn Khán...|   24|
|Ngõ 260 Đường Cầu...|   24|
|Ngõ 181 Đường Xuâ...|   24|
|Đường Nguyễn Văn ...|   24|
+--------------------+-----+
only showing top 20 rows



In [28]:
traffic_df = spark.read.format("delta").load(traffic_bronze_path)
traffic_df.show()

+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+
|           id|           road_name|  recordDatetime|density|occupancy|waitingTime|speed|sampledSeconds|traveltime|flow|entered|left|timeloss| processDatetime|
+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+
|-1008268818#0|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   1.89|     0.98|       13.0| 2.97|         21.75|     20.32|20.0|    1.0| 1.0|   16.85|2026-05-15T14:13|
|-1008268818#1|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.43|     0.22|        0.0|12.92|          0.68|       0.3|20.0|    1.0| 1.0|    0.04|2026-05-15T14:13|
|-1008268818#4|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.41|     0.21|        0.0|13.53|          0.87|       0.5|20.0|    1.0| 1.0|    0.02|2026-05-15T14:13|
|-1119291005#1|Ngõ 6 Phố Trần Qu...|2026

In [29]:
traffic_df.join(weather_df, "road_name", "inner").groupBy("road_name").count().show()

+--------------------+-----+
|           road_name|count|
+--------------------+-----+
| Phố Dương Quảng Hàm|   48|
|Ngõ 6 Trần Quốc Hoàn|   48|
|Ngõ 66 Dịch Vọng Hậu|  192|
| Đường Phạm Văn Đồng|  312|
|Ngõ 9 Phố Hoàng Q...|   48|
| Phố Trần Quốc Vượng|   24|
|      Ngõ 19 Duy Tân|   96|
|          Thành Thái|   24|
|      Ngõ 78 Duy Tân|  120|
|         Phố Tô Hiệu|   72|
|         Phố Duy Tân|  696|
|      Ngõ 86 Duy Tân|   24|
|Ngách 2 Ngõ 10 Ng...|   24|
|   Phố Phạm Tuấn Tài|  144|
|  Phố Hoàng Quán Chi|   96|
|Ngõ 381 Nguyễn Khang|  168|
|     Đường Phạm Hùng|  240|
|     Đường Xuân Thủy|  384|
|      Đường Cầu Giấy|  312|
|    Phố Khúc Thừa Dụ|  408|
+--------------------+-----+
only showing top 20 rows



In [30]:
from pyspark.sql import functions as F

WEATHER_INTERVAL_SECS = 3600

traffic_with_wtime = traffic_df.withColumn(
    "weather_time",
    from_unixtime(
        (F.unix_timestamp(F.col("recordDatetime"), "yyyy-MM-dd'T'HH:mm")
         / WEATHER_INTERVAL_SECS).cast("long") * WEATHER_INTERVAL_SECS,
        "yyyy-MM-dd'T'HH:mm"
    )
)

traffic_with_wtime.show(5)

+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+----------------+
|           id|           road_name|  recordDatetime|density|occupancy|waitingTime|speed|sampledSeconds|traveltime|flow|entered|left|timeloss| processDatetime|    weather_time|
+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+----------------+
|-1008268818#0|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   1.89|     0.98|       13.0| 2.97|         21.75|     20.32|20.0|    1.0| 1.0|   16.85|2026-05-15T14:13|2026-05-04T00:00|
|-1008268818#1|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.43|     0.22|        0.0|12.92|          0.68|       0.3|20.0|    1.0| 1.0|    0.04|2026-05-15T14:13|2026-05-04T00:00|
|-1008268818#4|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.41|     0.21|        0.0|13.53|          0.87|       0.5|

In [37]:
traffic_with_wtime.count()

305

In [32]:
weather_df = weather_df.withColumn(
    "datetime",
    F.regexp_replace(F.col("datetime"), r"^\d{4}-\d{2}-\d{2}", "2026-05-04")
)

In [33]:
w = weather_df.select(
    col("road_name").alias("w_road_name"),
    col("datetime").alias("w_datetime"),
    "temperature", "windspeed", "precipitation", "weather",
)

silver_df = traffic_with_wtime.join(
    w,
    on=(
        (traffic_with_wtime["road_name"] == w["w_road_name"]) &
        (traffic_with_wtime["weather_time"] == w["w_datetime"])
    ),
    how="inner",
).drop("w_road_name", "w_datetime", "weather_time")

In [34]:
silver_df.show()

+------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+-----+-------+----+--------+----------------+-----------+---------+-------------+--------+
|          id|           road_name|  recordDatetime|density|occupancy|waitingTime|speed|sampledSeconds|traveltime| flow|entered|left|timeloss| processDatetime|temperature|windspeed|precipitation| weather|
+------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+-----+-------+----+--------+----------------+-----------+---------+-------------+--------+
| 570600511#5|Ngõ 385 Đường Hoà...|2026-05-04T00:06|   0.42|     0.21|        0.0|13.16|          2.13|      1.76| 20.0|    1.0| 1.0|    0.10|2026-05-15T14:13|       17.0|      7.2|          0.0|Overcast|
| 570600511#4|Ngõ 385 Đường Hoà...|2026-05-04T00:06|   0.42|     0.21|        0.0|13.13|          0.81|      0.43| 20.0|    1.0| 1.0|    0.04|2026-05-15T14:13|       17.0|      7.2

In [38]:
silver_df.count()

291

In [35]:
from config import GCS_BUCKET

SILVER_PATH = f"gs://{GCS_BUCKET}/silver"

(
    silver_df
    .withColumn("date", F.to_date("recordDatetime", "yyyy-MM-dd'T'HH:mm"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("date")
    .save(SILVER_PATH)
)

print("Saved to Silver.")

Saved to Silver.
